# **Lab Report: Wing-Twist Optimisation Assignment**
<h3 align="RIGHT">Submitted by: Kalyana Raama Bheema<h3>

# **Gradient Descent: Wing-Twist Optimization**

Gradient descent is a method to find the minimum of a function $J(\boldsymbol{\alpha})$.\
The gradient $\nabla J$ points in the direction where $J$ increases the most, so we take a small step in the opposite direction:\
$$\boldsymbol{\alpha}_{k+1} = \boldsymbol{\alpha}_k - \eta\, \nabla J(\boldsymbol{\alpha}_k)$$\
$\eta$ is the learning rate (step size). We repeat this until the gradient is almost zero, because at the minimum $\nabla J = 0$.\
If $\eta$ is too small it takes many steps. If $\eta$ is too large the steps overshoot and $J$ blows up.

**Prandtl's lifting-line theory:**\
The spanwise coordinate is $y = -\frac{b}{2}\cos\theta$, with $\theta \in [0, \pi]$, and the circulation is written as a sine series:
$$\Gamma(\theta) = 2bV_\infty \sum_n A_n \sin(n\theta)$$\
For a symmetric wing only odd terms $n = 1, 3, 5, \dots$ are non-zero. The coefficients $A_n$ come from the monoplane equation at $N$ points on half the wing:
$$\sum_{j=1}^{N} A_{n_j} \sin(n_j\theta_i)\left[\frac{4b}{a_0 c(\theta_i)} + \frac{n_j}{\sin\theta_i}\right] = \alpha_i, \quad n_j = 2j-1, \quad \theta_i = \frac{i\pi}{2N}$$\
This is a linear system $M\mathbf{A} = \boldsymbol{\alpha}$. Once $\mathbf{A}$ is known:
$$C_L = \pi AR\, A_1, \qquad C_{D_i} = \pi AR \sum_j n_j A_{n_j}^2, \qquad e = \frac{C_L^2}{\pi AR\, C_{D_i}}$$

## **Problem Statement:**

*   **Wing twist optimization**: Twisting the wing changes the local angle of attack $\alpha(y)$, and with it the induced drag. We want the twist that gives the smallest induced drag for a required lift coefficient.

*   **Wing data**: Rectangular wing, $AR = 8$, $b = 1$, $c = b/AR$, $a_0 = 2\pi$ per radian, target $C_L^* = 0.5$.

*   **Design variables**: The $N = 12$ local angles $\boldsymbol{\alpha} = (\alpha_1, \dots, \alpha_N)$.

*   **Optimization problem**: The lift must equal the target, so a quadratic penalty is added to the drag:
$$\min_{\boldsymbol{\alpha} \in \mathbb{R}^N} J(\boldsymbol{\alpha}) = C_{D_i}(\boldsymbol{\alpha}) + \mu\,(C_L(\boldsymbol{\alpha}) - C_L^*)^2, \quad \mu = 10$$

*   **Elliptic loading (exact answer)**:
$$\alpha(\theta) = \frac{C_L}{\pi AR}\left[1 + \frac{4b}{a_0 c}\sin\theta\right]$$
This makes $A_n = 0$ for all $n \ge 3$, which gives $e = 1$.

In [1]:
# do not change line below
#from google.colab import files

# import required packages here
import numpy as np
import matplotlib.pyplot as plt

## **Task 1: Lifting-line function**

First a function that builds the matrix $M$:
$$M_{ij} = \sin(n_j\theta_i)\left[\frac{4b}{a_0 c} + \frac{n_j}{\sin\theta_i}\right]$$
Then a function that takes $\boldsymbol{\alpha}$, solves $M\mathbf{A} = \boldsymbol{\alpha}$ and returns $\mathbf{A}$, $C_L$, $C_{D_i}$ and $e$.

In [3]:
# wing data
AR = 8
b = 1
c = b / AR
a0 = 2 * np.pi
CL_target = 0.5

def matrix(N):
    theta = np.zeros(N)
    n = np.zeros(N)
    for i in range(N):
        theta[i] = (i + 1) * np.pi / (2 * N)
        n[i] = 2 * (i + 1) - 1
    M = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            M[i, j] = np.sin(n[j] * theta[i]) * (4 * b / (a0 * c) + n[j] / np.sin(theta[i]))
    return theta, n, M


def lifting_line(alpha):
    N = len(alpha)
    theta, n, M = matrix(N)
    A = np.linalg.solve(M, alpha)
    CL = np.pi * AR * A[0]
    CD = np.pi * AR * np.sum(n * A**2)
    e = CL**2 / (np.pi * AR * CD)
    return A, CL, CD, e